In [72]:
import numpy as np

In [73]:
import pandas as pd
labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        28177 non-null  int64
 1   Show              28177 non-null  str  
 2   EpId              28177 non-null  int64
 3   ClipId            28177 non-null  int64
 4   SEP28k-T          28177 non-null  str  
 5   SEP28k-D          28177 non-null  str  
 6   Prolongation      28177 non-null  int64
 7   Block             28177 non-null  int64
 8   SoundRep          28177 non-null  int64
 9   WordRep           28177 non-null  int64
 10  Interjection      28177 non-null  int64
 11  NoStutteredWords  28177 non-null  int64
 12  filepath          28177 non-null  str  
dtypes: int64(9), str(4)
memory usage: 2.8 MB


In [74]:

value_counts_table = (
    pd.concat(
        [df[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

value_counts_table

,Prolongation,Block,SoundRep,WordRep,Interjection
0,19631,16207,22563,23556,18519
1,5734,8600,3272,1851,3685
2,2022,2842,1479,1160,2595
3,790,528,863,1610,3378


In [75]:
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [76]:
df_filtered = df[
    df[labels].ge(2).any(axis=1) | df[labels].eq(0).all(axis=1)
].copy() #keep rows with >= 2, and rows with 0 (for fluent)


In [77]:
df_filtered.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath
0,0,HeStutters,0,0,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_0.wav
1,1,HeStutters,0,1,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_1.wav
2,2,HeStutters,0,2,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_2.wav
4,4,HeStutters,0,4,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_4.wav
6,6,HeStutters,0,6,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_6.wav


In [78]:

value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

print(value_counts_table)
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

   Prolongation  Block  SoundRep  WordRep  Interjection
0         14557  12790     15778    16207         12340
1          2657   3866      1906     1049          1713
2          2022   2842      1479     1160          2595
3           790    528       863     1610          3378


Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [79]:
#turn labels to (1,0) ground truths
df_filtered[labels] = (df_filtered[labels] >= 2).astype(np.float32)

In [80]:

df_filtered[labels]

,Prolongation,Block,SoundRep,WordRep,Interjection
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
28172,0.0,0.0,0.0,0.0,1.0
28173,0.0,0.0,1.0,0.0,0.0
28174,0.0,0.0,0.0,0.0,1.0
28175,0.0,0.0,0.0,0.0,1.0


In [81]:
df_filtered["fluent"] = df[labels].eq(0).all(axis=1).astype(int) #add a fluent flag if all disfluencies are 0.

In [82]:
sample_labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection', 'fluent']

In [83]:
value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in sample_labels],
        axis=1
    )
)

print(value_counts_table)


     Prolongation  Block  SoundRep  WordRep  Interjection  fluent
0.0         17214  16656     17684    17256         14053   14022
1.0          2812   3370      2342     2770          5973    6004


In [84]:
#now if we want to upsample, we need to make entries with ONLY that specific label to be upsampled
#similarly for down sample, we need to make entries with ONLY that label to be downsampled. 


In [85]:
#for now we can just downsample interjection and fluent to about 3k
df_fluent = df_filtered[df_filtered['fluent'] == 1]
df_other = df_filtered[df_filtered['fluent'] != 1]

df_label_down = df_fluent.sample(n=3000, random_state=20)

df_downsampled_fluent = pd.concat([df_label_down, df_other], ignore_index=True)

In [86]:
df_inter = df_downsampled_fluent[df_downsampled_fluent['Interjection'] == 1]
df_other = df_downsampled_fluent[df_downsampled_fluent['Interjection'] != 1]

df_label_down = df_inter.sample(n=3000, random_state=20)

df_downsampled = pd.concat([df_label_down, df_other], ignore_index=True)

In [87]:
value_counts_table = (
    pd.concat(
        [df_downsampled[col].value_counts().rename(col) for col in sample_labels],
        axis=1
    )
)

print(value_counts_table)

     Prolongation  Block  SoundRep  WordRep  Interjection  fluent
0.0         11561  10935     11853    11625         11049   11049
1.0          2488   3114      2196     2424          3000    3000


In [88]:
#At this point, it's going it be a multilabel classfication problem. Each input can have multiple ground truths that are 1 for the 5 disfluencies. 
#If the 5 labels are all 0, that means that it's fluent.

In [89]:
#Split entries based on SEP28k-T SEP
#copy so that they're their own objects
T_train_df = df_filtered[df_filtered["SEP28k-T"]== 'train'].copy()
T_dev_df = df_filtered[df_filtered["SEP28k-T"]== 'dev'].copy()
T_test_df = df_filtered[df_filtered["SEP28k-T"]== 'test'].copy()